# LimTOD horn threshold + rotation sanity diagnostics

This notebook is an **appendix notebook** to run after your main `limTOD_coordinate_consistent_clean_diagnostics.ipynb` notebook has already been executed.

It keeps the **real horn beam fixed** and tests whether changing the mapmaker/beam-support threshold allows the horn operator to include more low-declination / near-horizon pixels. It also checks whether uncertainty and residual problems remain confined to the patch boundary, rather than indicating a global LimTOD failure.

Main questions:

1. Does lowering the threshold increase the solved horn patch, especially toward lower declination?
2. Do the added lower-edge pixels have larger uncertainty, as expected?
3. Does the interior remain stable when the threshold is changed?
4. Does fast rotation still improve recovery relative to no rotation under the same threshold settings?


In [ ]:
# ============================================================
# BLOCK 0: Imports, required variables, and labels
# Run this after the main LimTOD notebook has already run.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
import inspect
import contextlib

required = [
    "real_beam",
    "sky_nside32",
    "reference_maps_full",
    "MAP_NSIDE",
    "SIM_NSIDE",
    "ANT_LAT_DEG",
    "AZIMUTH_DEG",
    "ELEVATION_DEG",
    "NORMALIZE_BEAM",
    "BEAM_TRUNCATE_FRAC",
    "NOISE_VARIANCE",
    "DT_SECONDS",
    "generate_TOD_sky",
    "build_mapper_with_matched_operator",
    "compute_hitmap_from_operator",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Run the main notebook first. Missing: {missing}")

# Use the same comparison reference you have been using.
THRESH_REF = "fwhm10"
THRESH_REG = 1e-4
THRESH_HIT_CUT = 1.0

if THRESH_REF not in reference_maps_full:
    raise KeyError(f"{THRESH_REF} not found in reference_maps_full. Available: {list(reference_maps_full.keys())}")

team_label_map = {
    "A_no_rotation_24h": "No rotation, 24 h",
    "B_slow_rotation_24h_one_turn_per_day": "Slow rotation, 24 h",
    "C_fast_rotation_24h_one_turn_per_hour": "Fast rotation, 24 h",
    "E_same_LST_12days_30deg_step_stacked": "Same LST, 12 days, 30° step",
    "G_same_LST_12days_fast_hourly_stacked": "Same LST, 12 days, fast hourly",
}

print("build_mapper_with_matched_operator signature:")
try:
    print(inspect.signature(build_mapper_with_matched_operator))
except Exception as exc:
    print("Could not inspect signature:", exc)

print("Current global threshold-like variables:")
for name in sorted(globals()):
    low = name.lower()
    if "thres" in low or "threshold" in low or "truncate" in low:
        val = globals()[name]
        if isinstance(val, (int, float, str, bool, type(None))):
            print(f"  {name} = {val}")


In [ ]:
# ============================================================
# BLOCK 1: General helpers
# ============================================================

def patch_to_full_threshold(patch, pix, nside, fill=hp.UNSEEN):
    full = np.full(hp.nside2npix(nside), fill, dtype=float)
    full[np.asarray(pix, dtype=int)] = np.asarray(patch, dtype=float).reshape(-1)
    return full


def metric_summary_threshold(recovered, truth, weighted_hit=None, hit_cut=0.0, mask=None):
    recovered = np.asarray(recovered, dtype=float).reshape(-1)
    truth = np.asarray(truth, dtype=float).reshape(-1)
    good = np.isfinite(recovered) & np.isfinite(truth)

    if weighted_hit is not None:
        weighted_hit = np.asarray(weighted_hit, dtype=float).reshape(-1)
        good &= np.isfinite(weighted_hit) & (weighted_hit > hit_cut)

    if mask is not None:
        mask = np.asarray(mask, dtype=bool).reshape(-1)
        good &= mask

    if good.sum() < 3:
        return {
            "n_pix": int(good.sum()),
            "rmse": np.nan,
            "bias": np.nan,
            "std": np.nan,
            "corr": np.nan,
            "median_abs_residual": np.nan,
            "p95_abs_residual": np.nan,
        }

    res = recovered[good] - truth[good]
    return {
        "n_pix": int(good.sum()),
        "rmse": float(np.sqrt(np.mean(res**2))),
        "bias": float(np.mean(res)),
        "std": float(np.std(res)),
        "corr": float(np.corrcoef(truth[good], recovered[good])[0, 1]),
        "median_abs_residual": float(np.median(np.abs(res))),
        "p95_abs_residual": float(np.percentile(np.abs(res), 95)),
    }


def selected_pixel_dec_ra(pix, nside):
    pix = np.asarray(pix, dtype=int)
    theta, phi = hp.pix2ang(nside, pix)
    dec = 90.0 - np.degrees(theta)
    ra = np.degrees(phi)
    return dec, ra


def selected_patch_summary(pix, weighted_hits=None, nside=None):
    if nside is None:
        nside = MAP_NSIDE
    pix = np.asarray(pix, dtype=int)
    dec, ra = selected_pixel_dec_ra(pix, nside)
    row = {
        "n_pix": int(len(pix)),
        "dec_min": float(np.min(dec)),
        "dec_p05": float(np.percentile(dec, 5)),
        "dec_median": float(np.median(dec)),
        "dec_p95": float(np.percentile(dec, 95)),
        "dec_max": float(np.max(dec)),
    }
    if weighted_hits is not None:
        wh = np.asarray(weighted_hits, dtype=float)
        row.update({
            "weighted_hit_min": float(np.nanmin(wh)),
            "weighted_hit_p05": float(np.nanpercentile(wh, 5)),
            "weighted_hit_median": float(np.nanmedian(wh)),
            "weighted_hit_p95": float(np.nanpercentile(wh, 95)),
            "weighted_hit_max": float(np.nanmax(wh)),
        })
    return row


def edge_mask_for_patch(pix, nside):
    """
    Edge pixels are selected pixels with at least one HEALPix neighbour outside the selected set.
    Interior pixels are selected pixels fully surrounded by selected pixels.
    """
    pix = np.asarray(pix, dtype=int)
    solved = set(pix.tolist())
    edge = np.zeros(len(pix), dtype=bool)
    for i, p in enumerate(pix):
        neigh = hp.get_all_neighbours(nside, int(p))
        neigh = neigh[neigh >= 0]
        if any(int(n) not in solved for n in neigh):
            edge[i] = True
    return edge


def edge_interior_metrics(result, hit_cut=0.0):
    pix = np.asarray(result["pix"], dtype=int)
    edge = edge_mask_for_patch(pix, MAP_NSIDE)
    interior = ~edge

    rows = []
    for region_name, mask in [("edge", edge), ("interior", interior), ("all", np.ones_like(edge, dtype=bool))]:
        m = metric_summary_threshold(
            result["recovered"],
            result["truth_patch"],
            result["weighted_hits"],
            hit_cut=hit_cut,
            mask=mask,
        )
        good = mask & np.isfinite(result["unc"]) & np.isfinite(result["weighted_hits"])
        if np.any(good):
            m.update({
                "unc_median": float(np.nanmedian(np.asarray(result["unc"])[good])),
                "unc_p95": float(np.nanpercentile(np.asarray(result["unc"])[good], 95)),
                "weighted_hit_median": float(np.nanmedian(np.asarray(result["weighted_hits"])[good])),
            })
        else:
            m.update({"unc_median": np.nan, "unc_p95": np.nan, "weighted_hit_median": np.nan})
        m["region"] = region_name
        rows.append(m)
    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# BLOCK 2: Threshold-aware mapper builder
# This is robust to different versions of your helper function.
# ============================================================

@contextlib.contextmanager
def temporary_global_values(updates):
    old = {}
    existed = {}
    for k, v in updates.items():
        existed[k] = k in globals()
        old[k] = globals().get(k, None)
        globals()[k] = v
    try:
        yield
    finally:
        for k in updates:
            if existed[k]:
                globals()[k] = old[k]
            else:
                globals().pop(k, None)


def build_mapper_threshold_aware(beam, LST_deg, az_deg, el_deg, selfrot_deg, select_threshold=None, beam_truncate_frac=None):
    """
    Tries to rebuild the mapper/operator with changed selection/truncation thresholds.

    This function supports two possibilities:
      1. build_mapper_with_matched_operator accepts threshold keywords.
      2. build_mapper_with_matched_operator reads global variables.

    If your helper ignores all these globals, then the selected pixel count will not change;
    the diagnostics will make that obvious.
    """
    sig = None
    try:
        sig = inspect.signature(build_mapper_with_matched_operator)
    except Exception:
        pass

    kwargs = {}
    if sig is not None:
        params = sig.parameters
        # Try common parameter names without assuming one exact notebook version.
        if select_threshold is not None:
            for nm in ["threshold", "select_threshold", "threshold_select_pix", "THRESHOLD_SELECT_PIX"]:
                if nm in params:
                    kwargs[nm] = select_threshold
                    break
        if beam_truncate_frac is not None:
            for nm in ["beam_truncate_frac", "beam_truncate_frac_thres", "truncate_frac_thres", "BEAM_TRUNCATE_FRAC"]:
                if nm in params:
                    kwargs[nm] = beam_truncate_frac
                    break

    global_updates = {}
    if select_threshold is not None:
        for nm in ["THRESHOLD_SELECT_PIX", "threshold_select_pix", "SELECT_THRESHOLD", "threshold"]:
            global_updates[nm] = select_threshold
    if beam_truncate_frac is not None:
        for nm in ["BEAM_TRUNCATE_FRAC", "beam_truncate_frac_thres", "TRUNCATE_FRAC_THRES", "truncate_frac_thres"]:
            global_updates[nm] = beam_truncate_frac

    with temporary_global_values(global_updates):
        mapper = build_mapper_with_matched_operator(
            beam,
            LST_deg,
            az_deg,
            el_deg,
            selfrot_deg,
            **kwargs,
        )
    return mapper


## Threshold sweep design

The primary sweep changes the **pixel-selection/support threshold** while keeping the same horn beam and the same scan. Lower threshold should include more weakly-supported pixels. If the setup is behaving sensibly, the added pixels should mostly appear near the boundary/lower-declination edge and should have larger uncertainty. The interior should remain comparatively stable.

This is not meant to make a final science map. It is a sanity test of the operator support and edge behaviour.


In [ ]:
# ============================================================
# BLOCK 3: Define scans and threshold grid
# ============================================================

# Use existing rotation clean scans if available. These are horn scans with real beam.
if "rotation_clean_results" not in globals():
    raise NameError("rotation_clean_results not found. Run your clean rotation test first.")

threshold_scan_cases = [
    "A_no_rotation_24h",
    "C_fast_rotation_24h_one_turn_per_hour",
]

# Add this if already available and you want to include it.
if "G_same_LST_12days_fast_hourly_stacked" in rotation_clean_results:
    threshold_scan_cases.append("G_same_LST_12days_fast_hourly_stacked")

threshold_scan_cases = [c for c in threshold_scan_cases if c in rotation_clean_results]

# Main selection-threshold sweep. Start with a small number to keep runtime sane.
# If your threshold variable has the opposite meaning, the n_pix trend will reveal it.
SELECT_THRESHOLDS_TO_TEST = [0.10, 0.05, 0.03, 0.02, 0.01, 0.005]

# Keep beam truncation fixed first. You can do a 2D sweep later if needed.
BEAM_TRUNCATE_TO_TEST = BEAM_TRUNCATE_FRAC

print("Scan cases:", threshold_scan_cases)
print("Selection thresholds:", SELECT_THRESHOLDS_TO_TEST)
print("Fixed beam truncation:", BEAM_TRUNCATE_TO_TEST)


In [ ]:
# ============================================================
# BLOCK 4: Run threshold sweep
# Real horn beam used for both TOD and mapmaker.
# No extra additive noise injected here.
# ============================================================

threshold_results = {}
threshold_rows = []
threshold_edge_rows = []

for scan_case in threshold_scan_cases:
    scan = rotation_clean_results[scan_case]["scan"]

    # Generate TOD once per scan. TOD is independent of mapmaker threshold.
    print("\n" + "#"*90)
    print("Generating TOD for scan:", scan_case)
    print("#"*90)

    tod = generate_TOD_sky(
        beam_map=real_beam,
        sky_map=sky_nside32,
        LST_deg_list=scan["LST_deg"],
        lat_deg=ANT_LAT_DEG,
        azimuth_deg_list=scan["az_deg"],
        elevation_deg_list=scan["el_deg"],
        selfrot_deg_list=scan["selfrot_deg"],
        nside_hires=SIM_NSIDE,
        normalize_beam=NORMALIZE_BEAM,
        truncate_frac_thres=BEAM_TRUNCATE_FRAC,
    )
    tod = np.asarray(tod, dtype=float).reshape(-1)

    print("TOD percentiles:", np.nanpercentile(tod, [0,1,5,50,95,99,100]))

    for select_thres in SELECT_THRESHOLDS_TO_TEST:
        label = f"{scan_case}__selectThres_{select_thres:g}"
        print("\n" + "="*80)
        print("Running threshold case:", label)
        print("="*80)

        mapper = build_mapper_threshold_aware(
            real_beam,
            scan["LST_deg"],
            scan["az_deg"],
            scan["el_deg"],
            scan["selfrot_deg"],
            select_threshold=select_thres,
            beam_truncate_frac=BEAM_TRUNCATE_TO_TEST,
        )

        pix = np.asarray(mapper.pixel_indices, dtype=int)
        A = np.asarray(mapper.Tsys_operators, dtype=float)
        hits, weighted_hits = compute_hitmap_from_operator(A)

        truth_patch = np.asarray(reference_maps_full[THRESH_REF][pix], dtype=float).reshape(-1)

        rec, unc = mapper(
            TOD_group=tod,
            dtime=scan["dt_seconds"],
            noise_variance=NOISE_VARIANCE,
            regularization=THRESH_REG,
            use_high_pass=False,
        )
        rec = np.asarray(rec, dtype=float).reshape(-1)
        unc = np.asarray(unc, dtype=float).reshape(-1)
        residual = rec - truth_patch

        try:
            svals = np.linalg.svd(A, compute_uv=False)
            cond = float(svals[0] / svals[-1])
            rank = int(np.linalg.matrix_rank(A))
        except Exception as exc:
            print("Could not compute SVD:", exc)
            cond = np.nan
            rank = np.nan

        result = {
            "label": label,
            "scan_case": scan_case,
            "scan": scan,
            "select_threshold": select_thres,
            "beam_truncate_frac": BEAM_TRUNCATE_TO_TEST,
            "tod": tod,
            "mapper": mapper,
            "A": A,
            "pix": pix,
            "hits": hits,
            "weighted_hits": weighted_hits,
            "truth_patch": truth_patch,
            "recovered": rec,
            "unc": unc,
            "residual": residual,
            "condition_number": cond,
            "rank": rank,
            "reference": THRESH_REF,
            "regularization": THRESH_REG,
        }
        threshold_results[label] = result

        patch_row = selected_patch_summary(pix, weighted_hits, nside=MAP_NSIDE)
        metrics_all = metric_summary_threshold(rec, truth_patch, weighted_hits, hit_cut=0.0)
        metrics_hit = metric_summary_threshold(rec, truth_patch, weighted_hits, hit_cut=THRESH_HIT_CUT)

        row = {
            "case": label,
            "scan_case": scan_case,
            "select_threshold": select_thres,
            "beam_truncate_frac": BEAM_TRUNCATE_TO_TEST,
            "n_tod": len(tod),
            "condition_number": cond,
            "rank": rank,
            **patch_row,
            "rmse_all": metrics_all["rmse"],
            "corr_all": metrics_all["corr"],
            "p95_abs_residual_all": metrics_all["p95_abs_residual"],
            "rmse_hit": metrics_hit["rmse"],
            "corr_hit": metrics_hit["corr"],
            "p95_abs_residual_hit": metrics_hit["p95_abs_residual"],
            "n_pix_hit": metrics_hit["n_pix"],
            "unc_median": float(np.nanmedian(unc)),
            "unc_p95": float(np.nanpercentile(unc, 95)),
        }
        threshold_rows.append(row)

        eim = edge_interior_metrics(result, hit_cut=0.0)
        eim["case"] = label
        eim["scan_case"] = scan_case
        eim["select_threshold"] = select_thres
        threshold_edge_rows.extend(eim.to_dict("records"))

        print("n_pix:", len(pix), "dec_min:", row["dec_min"], "rmse_hit:", row["rmse_hit"], "corr_hit:", row["corr_hit"])

threshold_df = pd.DataFrame(threshold_rows)
threshold_edge_df = pd.DataFrame(threshold_edge_rows)

display(threshold_df.sort_values(["scan_case", "select_threshold"], ascending=[True, False]))
display(threshold_edge_df.head())


In [ ]:
# ============================================================
# BLOCK 5: Threshold sweep summary plots
# ============================================================

scan_label_map = {
    "A_no_rotation_24h": "No rotation, 24 h",
    "C_fast_rotation_24h_one_turn_per_hour": "Fast rotation, 24 h",
    "G_same_LST_12days_fast_hourly_stacked": "Same LST, 12 days, fast hourly",
}

fig, ax = plt.subplots(2, 3, figsize=(16, 9))

for scan_case in threshold_df["scan_case"].unique():
    sub = threshold_df[threshold_df["scan_case"] == scan_case].sort_values("select_threshold")
    lab = scan_label_map.get(scan_case, scan_case)

    ax[0,0].plot(sub["select_threshold"], sub["n_pix"], marker="o", label=lab)
    ax[0,1].plot(sub["select_threshold"], sub["dec_min"], marker="o", label=lab)
    ax[0,2].plot(sub["select_threshold"], sub["condition_number"], marker="o", label=lab)
    ax[1,0].plot(sub["select_threshold"], sub["rmse_hit"], marker="o", label=lab)
    ax[1,1].plot(sub["select_threshold"], sub["corr_hit"], marker="o", label=lab)
    ax[1,2].plot(sub["select_threshold"], sub["unc_p95"], marker="o", label=lab)

for a in ax.ravel():
    a.set_xscale("log")
    a.invert_xaxis()
    a.grid(alpha=0.3)

ax[0,0].set_ylabel("Solved pixels")
ax[0,0].set_title("Patch size")
ax[0,1].set_ylabel("Minimum selected Dec [deg]")
ax[0,1].set_title("Does the patch extend lower?")
ax[0,2].set_ylabel("Condition number")
ax[0,2].set_yscale("log")
ax[0,2].set_title("Operator conditioning")
ax[1,0].set_ylabel(f"RMSE [K], hit>{THRESH_HIT_CUT}")
ax[1,0].set_title("Recovery error")
ax[1,1].set_ylabel(f"Correlation, hit>{THRESH_HIT_CUT}")
ax[1,1].set_title("Recovery correlation")
ax[1,2].set_ylabel("95% uncertainty [K-like]")
ax[1,2].set_title("Formal uncertainty")

for a in ax[1,:]:
    a.set_xlabel("Selection threshold")

ax[0,0].legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# BLOCK 6: Edge vs interior behaviour as threshold changes
# ============================================================

# Pivot edge/interior RMSE and uncertainty to check whether problems remain edge-localised.
edge_focus = threshold_edge_df[threshold_edge_df["region"].isin(["edge", "interior"])].copy()

display(edge_focus.sort_values(["scan_case", "select_threshold", "region"]))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for scan_case in edge_focus["scan_case"].unique():
    for region in ["edge", "interior"]:
        sub = edge_focus[(edge_focus["scan_case"] == scan_case) & (edge_focus["region"] == region)].sort_values("select_threshold")
        if len(sub) == 0:
            continue
        lab = f"{scan_label_map.get(scan_case, scan_case)} — {region}"
        ls = "-" if region == "interior" else "--"
        ax[0].plot(sub["select_threshold"], sub["rmse"], marker="o", linestyle=ls, label=lab)
        ax[1].plot(sub["select_threshold"], sub["unc_p95"], marker="o", linestyle=ls, label=lab)

for a in ax:
    a.set_xscale("log")
    a.invert_xaxis()
    a.grid(alpha=0.3)
    a.set_xlabel("Selection threshold")

ax[0].set_ylabel("RMSE [K]")
ax[0].set_title("Edge vs interior recovery error")
ax[1].set_ylabel("95% uncertainty [K-like]")
ax[1].set_title("Edge vs interior uncertainty")
ax[1].legend(fontsize=7, bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# BLOCK 7: Plot selected threshold maps
# Choose one threshold that expands patch but does not explode uncertainty.
# ============================================================

def plot_threshold_case(case_label, residual_percentile=95):
    r = threshold_results[case_label]
    pix = r["pix"]
    truth = r["truth_patch"]
    rec = r["recovered"]
    residual = r["residual"]
    unc = r["unc"]
    wh = r["weighted_hits"]

    truth_full = patch_to_full_threshold(truth, pix, MAP_NSIDE)
    rec_full = patch_to_full_threshold(rec, pix, MAP_NSIDE)
    res_full = patch_to_full_threshold(residual, pix, MAP_NSIDE)
    unc_full = patch_to_full_threshold(unc, pix, MAP_NSIDE)
    wh_full = patch_to_full_threshold(wh, pix, MAP_NSIDE)
    selected_full = patch_to_full_threshold(np.ones(len(pix)), pix, MAP_NSIDE)

    vmin, vmax = np.nanpercentile(truth[np.isfinite(truth)], [1, 99])
    rmax = np.nanpercentile(np.abs(residual[np.isfinite(residual)]), residual_percentile)
    umax = np.nanpercentile(unc[np.isfinite(unc)], 99)
    whmax = np.nanpercentile(wh[np.isfinite(wh)], 99)

    plt.figure(figsize=(16, 8.5))

    hp.mollview(truth_full, title=f"{r['reference']} reference", unit="K", min=vmin, max=vmax, sub=(2,3,1), badcolor="gray")
    hp.graticule()
    hp.mollview(rec_full, title="Recovered", unit="K", min=vmin, max=vmax, sub=(2,3,2), badcolor="gray")
    hp.graticule()
    hp.mollview(res_full, title="Recovered - reference", unit="K", min=-rmax, max=rmax, cmap="coolwarm", sub=(2,3,3), badcolor="gray")
    hp.graticule()
    hp.mollview(unc_full, title="Uncertainty", unit="K-like", min=0, max=umax, sub=(2,3,4), badcolor="gray")
    hp.graticule()
    hp.mollview(wh_full, title="Weighted hit", unit="sum(|A|)", min=0, max=whmax, sub=(2,3,5), badcolor="gray")
    hp.graticule()
    hp.mollview(selected_full, title="Solved pixel patch", min=0, max=1, sub=(2,3,6), badcolor="gray")
    hp.graticule()

    plt.suptitle(case_label, y=1.02, fontsize=12)
    plt.show()

    print("Patch summary:")
    display(pd.DataFrame([selected_patch_summary(pix, wh, MAP_NSIDE)]))
    print("Metrics hit cut:", metric_summary_threshold(rec, truth, wh, hit_cut=THRESH_HIT_CUT))
    display(edge_interior_metrics(r, hit_cut=0.0))


# Pick examples automatically: default-ish 0.05 and a lower threshold 0.01, if available.
example_thresholds = [0.05, 0.01]
for scan_case in ["C_fast_rotation_24h_one_turn_per_hour", "A_no_rotation_24h"]:
    for th in example_thresholds:
        label = f"{scan_case}__selectThres_{th:g}"
        if label in threshold_results:
            plot_threshold_case(label)


In [ ]:
# ============================================================
# BLOCK 8: Compare rotation at each threshold on common pixels
# This makes the claim 'rotation helps' fair at each threshold.
# ============================================================

def common_pixel_compare_for_two_cases(result_a, result_b, hit_cut=1.0):
    pix_a = np.asarray(result_a["pix"], dtype=int)
    pix_b = np.asarray(result_b["pix"], dtype=int)
    wh_a = np.asarray(result_a["weighted_hits"], dtype=float)
    wh_b = np.asarray(result_b["weighted_hits"], dtype=float)

    good_a = set(pix_a[np.isfinite(wh_a) & (wh_a > hit_cut)].tolist())
    good_b = set(pix_b[np.isfinite(wh_b) & (wh_b > hit_cut)].tolist())
    common_pix = sorted(good_a.intersection(good_b))

    ia = np.array([{int(p): i for i, p in enumerate(pix_a)}[p] for p in common_pix], dtype=int)
    ib = np.array([{int(p): i for i, p in enumerate(pix_b)}[p] for p in common_pix], dtype=int)

    rows = []
    for name, r, idx in [(result_a["scan_case"], result_a, ia), (result_b["scan_case"], result_b, ib)]:
        rec = np.asarray(r["recovered"])[idx]
        truth = np.asarray(r["truth_patch"])[idx]
        residual = rec - truth
        unc = np.asarray(r["unc"])[idx]
        wh = np.asarray(r["weighted_hits"])[idx]
        rows.append({
            "scan_case": name,
            "n_common_pix": len(common_pix),
            "rmse_common": float(np.sqrt(np.mean(residual**2))) if len(common_pix) else np.nan,
            "corr_common": float(np.corrcoef(truth, rec)[0,1]) if len(common_pix) > 2 else np.nan,
            "p95_abs_residual_common": float(np.percentile(np.abs(residual), 95)) if len(common_pix) else np.nan,
            "median_unc_common": float(np.median(unc)) if len(common_pix) else np.nan,
            "median_weighted_hit_common": float(np.median(wh)) if len(common_pix) else np.nan,
        })
    return pd.DataFrame(rows), np.asarray(common_pix, dtype=int)

rotation_threshold_rows = []

for th in SELECT_THRESHOLDS_TO_TEST:
    label_no = f"A_no_rotation_24h__selectThres_{th:g}"
    label_fast = f"C_fast_rotation_24h_one_turn_per_hour__selectThres_{th:g}"
    if label_no not in threshold_results or label_fast not in threshold_results:
        continue

    comp_df, common_pix = common_pixel_compare_for_two_cases(
        threshold_results[label_no],
        threshold_results[label_fast],
        hit_cut=THRESH_HIT_CUT,
    )
    comp_df["select_threshold"] = th
    rotation_threshold_rows.extend(comp_df.to_dict("records"))

rotation_threshold_common_df = pd.DataFrame(rotation_threshold_rows)
display(rotation_threshold_common_df)

# Improvement table: no rotation vs fast rotation at each threshold.
improve_rows = []
for th in SELECT_THRESHOLDS_TO_TEST:
    sub = rotation_threshold_common_df[rotation_threshold_common_df["select_threshold"] == th]
    if len(sub) != 2:
        continue
    no = sub[sub["scan_case"] == "A_no_rotation_24h"].iloc[0]
    fast = sub[sub["scan_case"] == "C_fast_rotation_24h_one_turn_per_hour"].iloc[0]
    improve_rows.append({
        "select_threshold": th,
        "n_common_pix": int(no["n_common_pix"]),
        "rmse_no_rotation": no["rmse_common"],
        "rmse_fast_rotation": fast["rmse_common"],
        "rmse_improvement_factor": no["rmse_common"] / fast["rmse_common"],
        "corr_no_rotation": no["corr_common"],
        "corr_fast_rotation": fast["corr_common"],
        "corr_gain": fast["corr_common"] - no["corr_common"],
        "p95_res_no_rotation": no["p95_abs_residual_common"],
        "p95_res_fast_rotation": fast["p95_abs_residual_common"],
    })

rotation_threshold_improvement_df = pd.DataFrame(improve_rows)
display(rotation_threshold_improvement_df.sort_values("select_threshold", ascending=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(rotation_threshold_improvement_df["select_threshold"], rotation_threshold_improvement_df["rmse_improvement_factor"], marker="o")
ax[0].axhline(1.0, linestyle="--", linewidth=1)
ax[0].set_xscale("log")
ax[0].invert_xaxis()
ax[0].set_xlabel("Selection threshold")
ax[0].set_ylabel("RMSE improvement factor: no rot / fast rot")
ax[0].set_title("Does fast rotation help at each threshold?")
ax[0].grid(alpha=0.3)

ax[1].plot(rotation_threshold_improvement_df["select_threshold"], rotation_threshold_improvement_df["corr_gain"], marker="o")
ax[1].axhline(0.0, linestyle="--", linewidth=1)
ax[1].set_xscale("log")
ax[1].invert_xaxis()
ax[1].set_xlabel("Selection threshold")
ax[1].set_ylabel("Correlation gain: fast - no rotation")
ax[1].set_title("Correlation gain from fast rotation")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## How to read the results

A threshold-lowering result supports the sanity check if:

- lowering the threshold increases `n_pix` and lowers `dec_min`, meaning the horn patch extends toward lower declinations;
- the new/low-declination pixels have lower weighted hit and higher uncertainty;
- edge pixels have higher uncertainty/residual than interior pixels;
- the interior does **not** become globally unstable;
- fast rotation still has lower RMSE / higher correlation than no rotation at the same threshold on common pixels.

A bad sign would be: lowering the threshold causes the interior uncertainty and residuals to explode everywhere. That would suggest a broader mapmaker/operator issue. A good sign is: instability is mainly boundary-localised, while the interior remains stable and rotation continues to help.
